# EUR/USD 4-Hour Historical Data — Dukascopy Download Notebook

This notebook downloads **EUR/USD 1-hour BID data** from Dukascopy, combines the yearly chunks, converts the data to **4-hour OHLCV candles**, validates the result, and saves the final dataset as:

`EURUSD_4H.csv`

## Project standard

- Pair: EUR/USD
- Source: Dukascopy
- Start: 2009-01-01
- End: 2026-09-02
- Source frequency: 1 Hour
- Final frequency: 4 Hours
- Timezone: UTC
- Offer side: BID
- Columns: Open, High, Low, Close, Volume


## 1. Install required packages

Run this cell once if `dukascopy-python` is not already installed.

In [ ]:
!pip install dukascopy-python pandas matplotlib

## 2. Import libraries

In [ ]:
from datetime import datetime
from time import sleep

import pandas as pd
import matplotlib.pyplot as plt
import dukascopy_python

from dukascopy_python.instruments import (
    INSTRUMENT_FX_MAJORS_EUR_USD
)

## 3. Define the EUR/USD download settings

We will download **1-hour candles** and later resample them into **4-hour candles**.

In [ ]:
START_DATE = datetime(2009, 1, 1)
END_DATE = datetime(2026, 9, 2)

instrument = INSTRUMENT_FX_MAJORS_EUR_USD
interval = dukascopy_python.INTERVAL_HOUR_1
offer_side = dukascopy_python.OFFER_SIDE_BID

print("Start date :", START_DATE)
print("End date   :", END_DATE)
print("Interval   : 1 Hour")
print("Offer side : BID")

## 4. Download the data year by year

Downloading in yearly chunks is safer than requesting the entire 2009–2026 period in one call.

In [ ]:
all_data = []

for year in range(2009, 2027):

    chunk_start = datetime(year, 1, 1)

    if year == 2026:
        chunk_end = END_DATE
    else:
        chunk_end = datetime(year + 1, 1, 1)

    print(
        f"Downloading {chunk_start.date()} "
        f"to {chunk_end.date()}..."
    )

    df = dukascopy_python.fetch(
        instrument=instrument,
        interval=interval,
        offer_side=offer_side,
        start=chunk_start,
        end=chunk_end,
        max_retries=5
    )

    if df is not None and not df.empty:
        all_data.append(df)
        print(f"Downloaded {len(df):,} rows")
    else:
        print("No data returned")

    sleep(1)

## 5. Combine all downloaded yearly chunks

In [ ]:
eurusd_h1 = pd.concat(all_data)

eurusd_h1.head()

## 6. Sort the time index

In [ ]:
eurusd_h1 = eurusd_h1.sort_index()

eurusd_h1.head()

## 7. Remove duplicate timestamps

This is a safety check because yearly chunks are being combined.

In [ ]:
eurusd_h1 = eurusd_h1[
    ~eurusd_h1.index.duplicated(keep="first")
]

print("Duplicate timestamps:", eurusd_h1.index.duplicated().sum())

## 8. Make sure timestamps use UTC

In [ ]:
if eurusd_h1.index.tz is None:
    eurusd_h1.index = eurusd_h1.index.tz_localize("UTC")
else:
    eurusd_h1.index = eurusd_h1.index.tz_convert("UTC")

print("Timezone:", eurusd_h1.index.tz)

## 9. Rename the columns

The package normally returns lowercase column names.  
We will standardize them to:

`Open, High, Low, Close, Volume`

In [ ]:
eurusd_h1 = eurusd_h1.rename(
    columns={
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
        "volume": "Volume"
    }
)

eurusd_h1.index.name = "DateTime"

eurusd_h1.head()

## 10. Inspect the 1-hour dataset

In [ ]:
print("Shape:", eurusd_h1.shape)
print("Start:", eurusd_h1.index.min())
print("End:", eurusd_h1.index.max())

eurusd_h1.info()

# Convert H1 data to H4 data

For OHLCV resampling:

- **Open** → first value
- **High** → maximum value
- **Low** → minimum value
- **Close** → last value
- **Volume** → sum


In [ ]:
eurusd_4h = eurusd_h1.resample(
    "4h",
    origin="start_day",
    label="left",
    closed="left"
).agg({
    "Open": "first",
    "High": "max",
    "Low": "min",
    "Close": "last",
    "Volume": "sum"
})

eurusd_4h.head()

## 11. Remove empty weekend periods

Forex does not trade continuously through the weekend.

We remove empty resampled periods instead of forward-filling them.

In [ ]:
eurusd_4h = eurusd_4h.dropna(
    subset=["Open", "High", "Low", "Close"]
)

eurusd_4h.head(10)

## 12. Validate the date range

In [ ]:
print("Start :", eurusd_4h.index.min())
print("End   :", eurusd_4h.index.max())
print("Rows  :", len(eurusd_4h))

## 13. Check the time gaps

The most common interval should be:

`0 days 04:00:00`

Longer gaps can occur because of weekends or market closures.

In [ ]:
eurusd_4h.index.to_series().diff().value_counts().head(10)

## 14. Check observations per year

In [ ]:
eurusd_4h.groupby(
    eurusd_4h.index.year
).size()

## 15. Check missing values

In [ ]:
eurusd_4h.isna().sum()

## 16. Check distinct available years

In [ ]:
eurusd_4h.index.year.unique()

## 17. Check distinct months in 2025

In [ ]:
eurusd_4h.loc["2025"].index.month_name().unique()

## 18. Display only 2025

In [ ]:
eurusd_4h.loc["2025"].head()

## 19. Display one month

Example: January 2025.

In [ ]:
eurusd_4h.loc["2025-01"].head()

## 20. Filter between two dates using a boolean mask

In [ ]:
eurusd_4h[
    (eurusd_4h.index >= "2025-01-01") &
    (eurusd_4h.index <= "2025-03-31")
].head()

## 21. Plot EUR/USD 4-hour closing price for 2025

In [ ]:
eurusd_2025 = eurusd_4h.loc["2025"]

plt.figure(figsize=(15, 6))

plt.plot(
    eurusd_2025.index,
    eurusd_2025["Close"]
)

plt.xlabel("Date")
plt.ylabel("EUR/USD")
plt.title("EUR/USD 4-Hour Close Price — 2025")

plt.xticks(rotation=45)
plt.grid()

plt.show()

## 22. Save the final clean 4-hour dataset

In [ ]:
eurusd_4h.to_csv("EURUSD_4H.csv")

print("Saved: EURUSD_4H.csv")

## 23. Load the saved file later

In future notebooks, you can load the clean dataset directly without downloading it again.

In [ ]:
eurusd = pd.read_csv(
    "EURUSD_4H.csv",
    parse_dates=["DateTime"],
    index_col="DateTime"
)

eurusd.head()

# Final validation block

Run this block whenever you want a quick health check of the dataset.

In [ ]:
print("Shape:")
print(eurusd_4h.shape)

print("\nDate Range:")
print(eurusd_4h.index.min())
print(eurusd_4h.index.max())

print("\nDuplicate Dates:")
print(eurusd_4h.index.duplicated().sum())

print("\nMissing Values:")
print(eurusd_4h.isna().sum())

print("\nMost Common Time Gaps:")
print(
    eurusd_4h.index
    .to_series()
    .diff()
    .value_counts()
    .head(10)
)

print("\nObservations Per Year:")
print(
    eurusd_4h.groupby(
        eurusd_4h.index.year
    ).size()
)

# Summary

This notebook performs the complete workflow:

1. Download EUR/USD 1-hour BID data from Dukascopy.
2. Download yearly chunks from 2009 to 2026.
3. Combine and sort the data.
4. Remove duplicate timestamps.
5. Standardize timestamps to UTC.
6. Rename OHLCV columns.
7. Resample H1 candles into H4 candles.
8. Remove empty weekend periods.
9. Validate frequency, missing values, years, and date range.
10. Save the final dataset as `EURUSD_4H.csv`.

You can now use `EURUSD_4H.csv` in your Time Series lessons and EUR/USD analysis project.
